In [ ]:
#@title 按這裡開始（先按 ▶）
print("✅ W10 出發！本週目標：自己寫斷詞與停用詞，訓練一個看得懂中文短評的情緒分類器")
print("本週要自己補五個空：cut()、clean()、測試集怎麼轉換、算分數、判錯的條件")
print("每一格由上往下依序按 ▶，跳著跑一定會出現 NameError")

# W10　中文短評情緒分析（電腦教室版）

一人一機，五個空格都要自己敲，不要互相複製貼上。

**開始之前先做這件事**：功能表「檔案 → 在雲端硬碟中儲存副本」。
直接開 GitHub 版是唯讀的，改了不會存下來，教室電腦重開機也會還原。

**這一本要交什麼**
- 寫完的 .ipynb（五個空格都填好）
- 錯誤分析表：五句判錯的句子＋你覺得為什麼會錯
- 截圖：濾停用詞前後的正確率

### 第 1 格：把全班短評讀進來，並自己寫斷詞函式 `cut()`

**這一格要做什麼**：裝好 jieba、讀進全班的短評，然後補完 `cut()` 裡的那一行。

**寫對了會看到什麼**：先印出「全班共 NN 筆短評」，再印出第一則短評被切開的一串詞。

看到 `<generator object ...>` 代表你用了 `jieba.cut()`，那回傳的是產生器；
本週要的是一個 list。安裝那一行大約要跑 30 秒，跑的時候不要切去別的分頁。

In [ ]:
#@title 第 1 格：讀資料＋自己寫 cut()
!pip install -q jieba
import pandas as pd, jieba

base = "https://docs.google.com/spreadsheets/d/"
sid  = ""   # ← 貼上老師上課公布的那一串英數字（留空白＝先用課程附的示範短評）
DEMO = "https://raw.githubusercontent.com/myliao2007/stust-course-1151/main/ai-intro-pc/data/demo_reviews.csv"  # ←投影片未含，執行所需
url  = base + sid + "/export?format=csv" if sid.strip() else DEMO   # ←投影片未含，執行所需
df = pd.read_csv(url)
df = df.rename(columns={"text": "短評", "label": "情緒"})   # ←投影片未含，執行所需（示範檔的欄名）
print("全班共", len(df), "筆短評")

jieba.add_word("滷肉飯")
jieba.add_word("南臺科技大學")

def cut(s):
    return ____              # ← 自己寫：回傳詞的 list
print(cut(df["短評"][0]))

### 第 2 格：自己建一份停用詞表，並寫出 `clean()`

停用詞就是「到處都有、幫不上忙」的那些字，先濾掉。

**這一格要做什麼**：補完 `clean()` 的那一行，把 `stop` 裡的字從斷詞結果中拿掉。
提示：一行 list 生成式就夠了。

**寫對了會看到什麼**：印出濾之前與濾之後兩串詞，最後一行說少了幾個詞，
通常會少掉三到五成。

**濾之前先想一下**：`不`、`沒` 千萬不要加進停用詞表，
濾掉會把「不好吃」變成「好吃」，答案就被你自己丟掉了。

In [ ]:
#@title 第 2 格：自己寫 clean()
stop = ["的", "了", "是", "我", "很", "也", "就", "都",
        "，", "。", "！", "？", " "]

def clean(s):
    words = cut(s)
    return ____              # ← 自己寫：把停用詞濾掉

raw = cut(df["短評"][0])
new = clean(df["短評"][0])
print("濾之前：", " / ".join(raw))
print("濾之後：", " / ".join(new))
print("少了", len(raw) - len(new), "個詞")

### 第 3 格：TF-IDF 加邏輯迴歸

**這一格要做什麼**：補兩個空。
第一個是「測試集只能轉換、不能再學一次字典」，第二個是評分用的方法名稱。

**寫對了會看到什麼**：印出一個 0 到 1 之間的正確率，通常落在 0.75 到 0.9。

**最容易錯的地方**：把測試集寫成 `fit_transform`，
那等於拿測試集重新學了一次字典，是標準的資料洩漏，期中考考過。

In [ ]:
#@title 第 3 格：訓練分類器
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X = [" ".join(clean(t)) for t in df["短評"]]
y = df["情緒"]
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.2, random_state=42)

v   = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b")
Vtr = v.fit_transform(Xtr)
Vte = ____                  # ← 自己寫：測試集只能轉換

m = LogisticRegression(max_iter=1000).fit(Vtr, ytr)
print("測試集正確率 =", round(m.____(Vte, yte), 3))

### 第 4 格：把判錯的句子列出來

錯誤分析是本週最有價值的一段，不要跳過。

**這一格要做什麼**：補上「判錯」的條件，把模型判錯的句子撈出來。

**寫對了會看到什麼**：印出「判錯 N 句，共 M 句」，接著印出前五句判錯的內容。
把這五句抄進作業的錯誤分析表，分成反話、錯字、新詞三類。

輸出的句子是斷詞後用空白接起來的，所以程式裡先把空白 `replace` 掉才好讀。

**只有 40 則示範短評時常常會判錯 0 句**，那是因為資料太少、太乾淨；
等老師公布 `sid`、換成全班兩百多筆真實資料再跑一次，就會有東西可以分析了。

In [ ]:
#@title 第 4 格：錯誤分析
pred = m.predict(Vte)
ans  = list(yte)
txt  = list(Xte)

bad = []
for i in range(len(txt)):
    if ____:                # ← 自己寫：判錯的條件
        bad.append((txt[i], ans[i], pred[i]))

print("判錯", len(bad), "句，共", len(txt), "句")
for s, a, p in bad[:5]:
    print("正解", a, "／判成", p, "：", s.replace(" ", ""))

### 第 5 格（進階）：換分類器、加 bigram

同一份資料換兩種做法，看分數與特徵數怎麼變。這一格沒有空格，直接執行。

**寫對了會看到什麼**：三個數字——LinearSVC 的分數、加了 bigram 的分數，
以及特徵數從幾個變成幾個（通常變三到四倍）。

`ngram_range=(1, 2)` 代表把相鄰兩個詞也當成一個特徵，
「不 好吃」變成一個特徵之後，反話就抓得到了。
資料量小的時候分數不一定會變高，那本身就是可以寫進作業的結果。

In [ ]:
#@title 第 5 格（進階）：LinearSVC 與 bigram
from sklearn.svm import LinearSVC

m2 = LinearSVC().fit(Vtr, ytr)
print("LinearSVC =", round(m2.score(Vte, yte), 3))

v2   = TfidfVectorizer(ngram_range=(1, 2),
                       token_pattern=r"(?u)\b\w+\b")
Vtr2 = v2.fit_transform(Xtr)
Vte2 = v2.transform(Xte)
m3 = LogisticRegression(max_iter=1000).fit(Vtr2, ytr)
print("加 bigram =", round(m3.score(Vte2, yte), 3))
print("特徵數：", len(v.vocabulary_), "→", len(v2.vocabulary_))

### 收工：延伸挑戰與繳交

- **A**：把 `test_size` 從 0.2 改成 0.4 再跑一次第 3 格，兩次正確率都記下來。
- **B**：把「不」「沒」故意加進 `stop` 再訓練一次，正確率掉了多少？判錯的句子變成哪一種？
- **C**：挑一句被判錯的短評，用兩句話說出模型為什麼會錯，再寫出你打算怎麼補救。

**常見狀況**：`NameError: cut` 是前面那一格沒執行；
說找不到 jieba 是執行階段被重新啟動了，回頭重跑第 1 格；
正確率高得離譜先看 `df["情緒"].value_counts()`，八成是正負面筆數差太多。

In [ ]:
#@title 收工檢查（直接按 ▶）
print("本週要交：寫完的 .ipynb、錯誤分析表五句、濾停用詞前後的正確率截圖")
print("檔名：AI導論_W10_學號_姓名，繳交期限：下次上課前一天 23:59")
print("交之前先確認：五個空格都填好，而且整本由上往下重跑一次不會報錯")

---

<details>
<summary>參考解（五個空格都自己試過再打開）</summary>

```python
# 第 1 格
def cut(s):
    return jieba.lcut(s)

# 第 2 格
def clean(s):
    words = cut(s)
    return [w for w in words if w not in stop]

# 第 3 格
Vte = v.transform(Xte)
print("測試集正確率 =", round(m.score(Vte, yte), 3))

# 第 4 格
    if pred[i] != ans[i]:
```

為什麼是這樣寫：

- `jieba.lcut()` 直接回傳 list；`jieba.cut()` 回傳的是產生器，印出來會看到奇怪的物件。
- 濾停用詞就是一行 list 生成式：留下不在 `stop` 裡的詞。
- 測試集只能 `transform`。寫成 `fit_transform` 等於用測試集重新學一次字典，是資料洩漏。
- 判錯的條件就是「預測 ≠ 正解」。

</details>